# Current roster pre-Unity package (CH101~CH105)

최신 승인 2D 시트를 기준으로 5명 전부의 Blender 사전-Unity 산출물을 만듭니다.
공통 처리: blockout, UV/material 점검, 22-bone rig prototype, socket, rigid blockout weights, pose preview, FBX, JSON validation.
final skinning, Unity Humanoid mapping, Android proof, Gate B 승인은 이 노트북의 범위가 아닙니다.

In [ ]:
get_ipython().system('apt-get -qq update')
get_ipython().system('apt-get -qq install -y blender xvfb')
import subprocess
subprocess.run(['blender', '--version'], check=True)

In [ ]:
import json, os, shutil, subprocess
from pathlib import Path
from google.colab import files
REPO_URL = 'https://github.com/siri2677/re-camp.git'
TOOLS_REPO_URL = 'https://github.com/siri2677/re-camp-blender.git'
TOOLS_BRANCH = 'agent/current-roster-pre-unity'
ART_BRANCH = 'art/current-roster-gate-a-ch102'
ART_DIR = Path('/content/re-camp')
TOOLS_DIR = Path('/content/re-camp-blender')
OUTPUT_ROOT = Path('/content/re-camp-output/current-roster-pre-unity-v001')
if ART_DIR.exists(): shutil.rmtree(ART_DIR)
if TOOLS_DIR.exists(): shutil.rmtree(TOOLS_DIR)
subprocess.run(['git', 'clone', '--branch', ART_BRANCH, '--single-branch', REPO_URL, str(ART_DIR)], check=True)
subprocess.run(['git', 'clone', '--depth', '1', '--branch', TOOLS_BRANCH, TOOLS_REPO_URL, str(TOOLS_DIR)], check=True)
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
print('art HEAD:', subprocess.check_output(['git', '-C', str(ART_DIR), 'rev-parse', 'HEAD'], text=True).strip())
print('tools HEAD:', subprocess.check_output(['git', '-C', str(TOOLS_DIR), 'rev-parse', 'HEAD'], text=True).strip())

In [ ]:
characters = [
    {'id': 'CH101', 'name': 'Rin', 'asset': 'art_refs/characters/rin/concept/CH101_Rin_CharacterSheet_APPROVED_v001.png', 'commit': 'ae111815f356f105a170bf1c473ae3cd086fee4c', 'builder': 'build_blockout.py'},
    {'id': 'CH102', 'name': 'Mao', 'asset': 'art_refs/characters/mao/concept/CH102_Mao_CharacterSheet_APPROVED_v001.png', 'commit': 'c01a4e2d90919872997dccf46679cf8ab10f1e87', 'builder': 'build_roster.py'},
    {'id': 'CH103', 'name': 'Nozomi', 'asset': 'art_refs/characters/nozomi/concept/CH103_Nozomi_CharacterSheet_APPROVED_v001.png', 'commit': 'f231ed5a4491eaa274237b7a2c8d4911d538b0ab', 'builder': 'build_roster.py'},
    {'id': 'CH104', 'name': 'Shion', 'asset': 'art_refs/characters/shion/concept/CH104_Shion_CharacterSheet_APPROVED_v001.png', 'commit': '2dcb002f2691006008d0c20fa8157cbdd7d52538', 'builder': 'build_roster.py'},
    {'id': 'CH105', 'name': 'Akari', 'asset': 'art_refs/characters/akari/concept/CH105_Akari_CharacterSheet_APPROVED_v001.png', 'commit': 'd876bbc0c2eef7e9e549de274c15b3ab190ad6ce', 'builder': 'build_roster.py'},
]
for row in characters:
    asset_path = ART_DIR / row['asset']
    if not asset_path.is_file(): raise FileNotFoundError(asset_path)
    print(row['id'], row['name'], row['commit'], asset_path)

In [ ]:
import json, subprocess
manifest = {'roster': [], 'status': 'PASS', 'source_branch': ART_BRANCH}
for row in characters:
    output_dir = OUTPUT_ROOT / row['id']
    build_script = TOOLS_DIR / 'scripts' / 'blender' / row['builder']
    validate_script = TOOLS_DIR / 'scripts' / 'blender' / 'validate_asset.py'
    blend_path = output_dir / f"{row['id']}_Blockout_REVIEW_v007.blend"
    report_path = output_dir / 'reports' / f"{row['id']}_validation.json"
    source_asset = ART_DIR / row['asset']
    build_cmd = ['xvfb-run', '-a', 'blender', '-b', '--python', str(build_script), '--', '--character', row['id'], '--source-asset', str(source_asset), '--source-commit', row['commit'], '--output-dir', str(output_dir), '--render', '--export-fbx']
    print('BUILD', row['id'])
    build_run = subprocess.run(build_cmd, capture_output=True, text=True)
    print(build_run.stdout[-6000:])
    if build_run.returncode != 0 or not blend_path.is_file():
        print(build_run.stderr[-6000:])
        raise RuntimeError(f"Build failed or blend missing for {row['id']}: {build_run.returncode}")
    validate_cmd = ['xvfb-run', '-a', 'blender', '-b', '--python', str(validate_script), '--', '--character', row['id'], '--blend', str(blend_path), '--report', str(report_path)]
    print('VALIDATE', row['id'])
    validate_run = subprocess.run(validate_cmd, capture_output=True, text=True)
    print(validate_run.stdout[-6000:])
    if validate_run.returncode != 0 or not report_path.is_file():
        print(validate_run.stderr[-6000:])
        raise RuntimeError(f"Validation failed or report missing for {row['id']}: {validate_run.returncode}")
    validation = json.loads(report_path.read_text())
    manifest['roster'].append({'id': row['id'], 'name': row['name'], 'source_asset': row['asset'], 'source_commit': row['commit'], 'validation': validation})
    if validation.get('status') != 'PASS': manifest['status'] = 'FAIL'
manifest_path = OUTPUT_ROOT / 'current_roster_pre_unity_manifest.json'
manifest_path.write_text(json.dumps(manifest, indent=2))
print(json.dumps({'status': manifest['status'], 'characters': [item['id'] for item in manifest['roster']]}, indent=2))
if manifest['status'] != 'PASS': raise RuntimeError('One or more roster validations failed')

In [ ]:
qa_script = TOOLS_DIR / 'scripts' / 'qa' / 'prepare_pre_unity_package.py'
qa_contact_sheet = OUTPUT_ROOT / 'qa' / 'current_roster_visual_qa_contact_sheet.png'
qa_manifest = OUTPUT_ROOT / 'pre_unity_package_manifest.json'
qa_cmd = ['python', str(qa_script), '--output-root', str(OUTPUT_ROOT), '--art-root', str(ART_DIR), '--manifest', str(manifest_path), '--contact-sheet', str(qa_contact_sheet)]
qa_run = subprocess.run(qa_cmd, capture_output=True, text=True)
print(qa_run.stdout)
if qa_run.returncode != 0 or not qa_manifest.is_file() or not qa_contact_sheet.is_file():
    print(qa_run.stderr[-6000:])
    raise RuntimeError(f'Package QA failed: {qa_run.returncode}')

In [ ]:
archive = shutil.make_archive('/content/re-camp-current-roster-pre-unity-v001', 'zip', OUTPUT_ROOT)
archive_hash = __import__('hashlib').sha256(Path(archive).read_bytes()).hexdigest()
Path(str(archive) + '.sha256').write_text(archive_hash + '  ' + Path(archive).name + '\n')
print('ZIP:', archive)
print('ZIP SHA256:', archive_hash)
files.download(archive)
files.download(str(archive) + '.sha256')